# Komposit Rainfall El Nino

Notebook ini mengambil hanya bagian komposit El Niño dari `rainfall_analysis_v3.ipynb` dan menyesuaikannya dengan rules komposit yang sudah dipakai di notebook `komposit_*`.

Plot komposit hanya untuk domain Maritime Continent. Semua output disimpan ke `results/analisis_komposit_26-19/komposit_rainfall_elnino/`.

In [ ]:
from pathlib import Path

import dask
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
from scipy import stats
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from IPython.display import display

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'

RAINFALL_PATH = Path('/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/mswep-monthly/mswep_monthly_combined.nc')
NINO34_PATH = Path('/Users/rizzie/TugasAkhir/data_processing/external/ClimateData/index-monthly/nino34.anom.csv')
RESULTS_DIR = Path('/Users/rizzie/TugasAkhir/data_processing/results/analisis_komposit_26-19/komposit_rainfall_elnino')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR = Path('/Users/rizzie/TugasAkhir/data_processing/data/intermediate/komposit_rainfall_elnino')
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_PATH = INTERMEDIATE_DIR / 'rainfall_elnino_mc_composites_significance.nc'

mc_extent = [80, 160, -20, 20]
mc_xticks = np.arange(90, 161, 20)
mc_yticks = np.arange(-20, 21, 10)

print('Results folder:', RESULTS_DIR)

In [ ]:
mswep_path = RAINFALL_PATH
ds_mswep = xr.open_dataset(mswep_path, chunks={'time': 12})['precipitation']
ds_mswep = ds_mswep.assign_coords(lon=(ds_mswep.lon % 360)).sortby('lon')
ds_mswep = ds_mswep.sel(
    time=slice('1980-12-01', '2025-02-01'),
    lat=slice(21, -21),
    lon=slice(70, 290),
)
month_mask = ds_mswep.time.dt.month.isin([12, 1, 2])
djf_year = xr.where(ds_mswep.time.dt.month == 12, ds_mswep.time.dt.year + 1, ds_mswep.time.dt.year)
ds_mswep = ds_mswep.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))

nino34_column = "   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/"
df_nino34 = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2025-03-01']
df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int8')
df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)

print('Rainfall time coverage:', str(ds_mswep.time.values[0])[:10], 'to', str(ds_mswep.time.values[-1])[:10])
print('Niño3.4 DJF coverage:', int(df_nino34['djf_year'].min()), 'to', int(df_nino34['djf_year'].max()))

In [ ]:
full_years = np.arange(1981, 2026)
base_years = np.arange(1991, 2021)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2026)

rain_clim = ds_mswep.sel(time=ds_mswep.djf_year.isin(full_years))
rain_base = ds_mswep.sel(time=ds_mswep.djf_year.isin(base_years))
rain_past = rain_clim.sel(time=rain_clim.djf_year.isin(past_years))
rain_recent = rain_clim.sel(time=rain_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()

period_coord = xr.DataArray(
    np.where(rain_clim.djf_year <= 2006, 'past', 'recent'),
    coords={'time': rain_clim.time},
    dims='time',
    name='period',
)
rain_period = rain_clim.assign_coords(period=period_coord)

base_plot, full_plot = dask.compute(
    rain_base.mean('time'),
    rain_clim.mean('time'),
)
period_means = rain_period.groupby('period').mean('time').compute()

base_plot = base_plot.transpose('lat', 'lon')
full_plot = full_plot.transpose('lat', 'lon')
past_plot = period_means.sel(period='past').transpose('lat', 'lon')
recent_plot = period_means.sel(period='recent').transpose('lat', 'lon')
diff_plot = recent_plot - past_plot

print('Base years:', int(base_years[0]), 'to', int(base_years[-1]))
print('Full years:', int(full_years[0]), 'to', int(full_years[-1]))
print('Past years:', int(past_years[0]), 'to', int(past_years[-1]))
print('Recent years:', int(recent_years[0]), 'to', int(recent_years[-1]))


In [ ]:
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')[nino34_column].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf[nino34_column] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())

print('El Nino DJF years (1981-2025):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2025):', elnino_years_recent)

if INTERMEDIATE_PATH.exists():
    cached_ds = xr.load_dataset(INTERMEDIATE_PATH)
    rain_clim_elnino = cached_ds['komposit_full']
    rain_past_elnino = cached_ds['komposit_past']
    rain_recent_elnino = cached_ds['komposit_recent']
    rain_clim_elnino_anom = cached_ds['anomali_full']
    rain_past_elnino_anom = cached_ds['anomali_past']
    rain_recent_elnino_anom = cached_ds['anomali_recent']
    diff_elnino_anom = cached_ds['anomali_diff']
    print('Loaded intermediate composites from:', INTERMEDIATE_PATH)
else:
    rain_clim_elnino = rain_clim.sel(time=rain_clim.djf_year.isin(elnino_years_clim))
    rain_past_elnino = rain_past.sel(time=rain_past.djf_year.isin(elnino_years_past))
    rain_recent_elnino = rain_recent.sel(time=rain_recent.djf_year.isin(elnino_years_recent))

    rain_clim_elnino, rain_past_elnino, rain_recent_elnino = dask.compute(
        rain_clim_elnino.mean('time'),
        rain_past_elnino.mean('time'),
        rain_recent_elnino.mean('time'),
    )

    rain_clim_elnino = rain_clim_elnino.transpose('lat', 'lon')
    rain_past_elnino = rain_past_elnino.transpose('lat', 'lon')
    rain_recent_elnino = rain_recent_elnino.transpose('lat', 'lon')

    rain_clim_elnino_anom = (rain_clim_elnino - base_plot).reset_coords(drop=True)
    rain_past_elnino_anom = (rain_past_elnino - base_plot).reset_coords(drop=True)
    rain_recent_elnino_anom = (rain_recent_elnino - base_plot).reset_coords(drop=True)
    diff_elnino_anom = (rain_recent_elnino_anom - rain_past_elnino_anom).reset_coords(drop=True)

In [ ]:
def add_stippling(ax, sig_mask, block=8, size=15, alpha=0.7):
    sig = sig_mask.fillna(False).astype(np.int8)
    sig_plot = (sig.coarsen(lat=block, lon=block, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size == 0:
        return
    ax.scatter(
        sig_plot['lon'].values[xx],
        sig_plot['lat'].values[yy],
        s=size,
        c='k',
        marker='.',
        linewidths=0,
        alpha=alpha,
        transform=ccrs.PlateCarree(),
        zorder=4,
    )


def plot_and_save(data, title, cmap, levels, ticks, cbar_label, extend, extent, xticks, yticks, outfile, figsize, ticksize, title_size, cbar_size, sig_mask=None):
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=8, size=15, alpha=0.7)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=ticksize)
    ax.set_title(title, fontsize=title_size, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=cbar_size, labelpad=10)
    cbar.ax.tick_params(labelsize=cbar_size)

    fig.savefig(RESULTS_DIR / outfile, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


def _compute_array(data):
    if hasattr(data, 'compute'):
        data = data.compute()
    return np.asarray(data)


def _pvalue_map(pvalues, template):
    return xr.DataArray(pvalues, coords=template.coords, dims=template.dims)


def _ttest_1samp_map(sample, popmean, template):
    pvalues = stats.ttest_1samp(_compute_array(sample), _compute_array(popmean), axis=0, nan_policy='omit').pvalue
    return _pvalue_map(pvalues, template)


def _welch_ttest_map(sample_a, sample_b, template):
    pvalues = stats.ttest_ind(_compute_array(sample_a), _compute_array(sample_b), axis=0, equal_var=False, nan_policy='omit').pvalue
    return _pvalue_map(pvalues, template)


## Seleksi event El Nino

Bagian ini mengecek event El Nino yang lolos threshold `>= 0.5` untuk P1 dan P2 sebelum masuk ke komposit hujan.
Tabel menampilkan DJF Niño 3.4 tiap event, lalu ringkasan mean/median/min/max dan jumlah event per periode.

In [ ]:
event_p1 = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold].copy()
event_p2 = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold].copy()

event_p1['period'] = 'P1'
event_p2['period'] = 'P2'

event_table = pd.concat([event_p1, event_p2], ignore_index=True)[['period', 'djf_year', nino34_column]].copy()
event_table = event_table.rename(columns={nino34_column: 'nino34'})
event_table['nino34'] = pd.to_numeric(event_table['nino34'], errors='coerce')

selection_summary = event_table.groupby('period')['nino34'].agg(['count', 'mean', 'median', 'min', 'max']).reset_index()

display(event_table.sort_values(['period', 'djf_year']).reset_index(drop=True))
display(selection_summary)

print('P1 event years:', event_p1['djf_year'].tolist())
print('P2 event years:', event_p2['djf_year'].tolist())
print('P1 count:', len(event_p1))
print('P2 count:', len(event_p2))

event_table.to_csv(RESULTS_DIR / 'elnino_event_selection_p1_p2.csv', index=False)
selection_summary.to_csv(RESULTS_DIR / 'elnino_event_selection_summary.csv', index=False)


In [ ]:
p1_vals = event_table.loc[event_table['period'] == 'P1', 'nino34'].dropna().to_numpy()
p2_vals = event_table.loc[event_table['period'] == 'P2', 'nino34'].dropna().to_numpy()
all_vals = event_table['nino34'].dropna().to_numpy()
hist_min = np.floor(all_vals.min() * 4) / 4 if all_vals.size else 0.5
hist_max = np.ceil(all_vals.max() * 4) / 4 if all_vals.size else 2.5
hist_bins = np.arange(hist_min, hist_max + 0.25, 0.25)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
box = axes[0].boxplot([p1_vals, p2_vals], labels=['P1', 'P2'], patch_artist=True, showmeans=True)
for patch, color in zip(box['boxes'], ['#4C78A8', '#F58518']):
    patch.set_facecolor(color)
for median in box['medians']:
    median.set_color('black')
axes[0].axhline(elnino_threshold, color='gray', linestyle='--', linewidth=1, alpha=0.8)
axes[0].set_title('Boxplot DJF Niño3.4 event El Nino')
axes[0].set_ylabel('DJF Niño3.4')
axes[0].grid(True, linestyle='--', alpha=0.3)

axes[1].hist([p1_vals, p2_vals], bins=hist_bins, label=['P1', 'P2'], color=['#4C78A8', '#F58518'], alpha=0.65, edgecolor='white')
axes[1].axvline(elnino_threshold, color='gray', linestyle='--', linewidth=1, alpha=0.8)
axes[1].set_title('Histogram DJF Niño3.4 event El Nino')
axes[1].set_xlabel('DJF Niño3.4')
axes[1].set_ylabel('Jumlah event')
axes[1].legend(frameon=False)
axes[1].grid(True, linestyle='--', alpha=0.3)

fig.tight_layout()
fig.savefig(RESULTS_DIR / 'elnino_event_selection_boxplot_hist.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 5))
p1_years = event_table.loc[event_table['period'] == 'P1', 'djf_year'].to_numpy()
p2_years = event_table.loc[event_table['period'] == 'P2', 'djf_year'].to_numpy()
ax.scatter(p1_years, p1_vals, s=55, color='#4C78A8', edgecolors='white', linewidths=0.6, label='P1')
ax.scatter(p2_years, p2_vals, s=55, color='#F58518', edgecolors='white', linewidths=0.6, label='P2')
ax.axhline(elnino_threshold, color='gray', linestyle='--', linewidth=1, alpha=0.8)
ax.axvline(2006.5, color='gray', linestyle=':', linewidth=1, alpha=0.8)
ax.set_title('Scatter DJF Niño3.4 event El Nino')
ax.set_xlabel('DJF year')
ax.set_ylabel('DJF Niño3.4')
ax.legend(frameon=False)
ax.grid(True, linestyle='--', alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'elnino_event_selection_scatter_year_value.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)


In [ ]:
if INTERMEDIATE_PATH.exists():
    cached_ds = xr.load_dataset(INTERMEDIATE_PATH)
    rain_clim_elnino_p = cached_ds['pvalue_full']
    rain_past_elnino_p = cached_ds['pvalue_past']
    rain_recent_elnino_p = cached_ds['pvalue_recent']
    rain_diff_elnino_p = cached_ds['pvalue_diff']
    rain_clim_elnino_sig = cached_ds['signif_full'].astype(bool)
    rain_past_elnino_sig = cached_ds['signif_past'].astype(bool)
    rain_recent_elnino_sig = cached_ds['signif_recent'].astype(bool)
    rain_diff_elnino_sig = cached_ds['signif_diff'].astype(bool)
    print('Loaded intermediate significance from:', INTERMEDIATE_PATH)
else:
    rain_clim_elnino_samples = rain_clim.sel(time=rain_clim.djf_year.isin(elnino_years_clim)).transpose('time', 'lat', 'lon')
    rain_past_elnino_samples = rain_past.sel(time=rain_past.djf_year.isin(elnino_years_past)).transpose('time', 'lat', 'lon')
    rain_recent_elnino_samples = rain_recent.sel(time=rain_recent.djf_year.isin(elnino_years_recent)).transpose('time', 'lat', 'lon')

    rain_clim_elnino_samples, rain_past_elnino_samples, rain_recent_elnino_samples = dask.compute(
        rain_clim_elnino_samples,
        rain_past_elnino_samples,
        rain_recent_elnino_samples,
    )

    rain_clim_elnino_p = _ttest_1samp_map(rain_clim_elnino_samples, base_plot, base_plot)
    rain_past_elnino_p = _ttest_1samp_map(rain_past_elnino_samples, base_plot, base_plot)
    rain_recent_elnino_p = _ttest_1samp_map(rain_recent_elnino_samples, base_plot, base_plot)
    rain_diff_elnino_p = _welch_ttest_map(rain_recent_elnino_samples, rain_past_elnino_samples, base_plot)

    rain_clim_elnino_sig = rain_clim_elnino_p < 0.05
    rain_past_elnino_sig = rain_past_elnino_p < 0.05
    rain_recent_elnino_sig = rain_recent_elnino_p < 0.05
    rain_diff_elnino_sig = rain_diff_elnino_p < 0.05


## Komposit El Nino - Maritime Continent

In [ ]:
plots = [
    (rain_clim_elnino_anom, rain_clim_elnino_sig, 'Komposit El Nino DJF 1981-2025, anomaly vs DJF 1991-2020', 'RdBu', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'Anomali curah hujan vs DJF 1991-2020 (mm/bulan)', 'both', 'rainfall_elnino_mc_full.png'),
    (rain_past_elnino_anom, rain_past_elnino_sig, 'Komposit El Nino DJF 1981-2006, anomaly vs DJF 1991-2020', 'RdBu', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'Anomali curah hujan vs DJF 1991-2020 (mm/bulan)', 'both', 'rainfall_elnino_mc_past.png'),
    (rain_recent_elnino_anom, rain_recent_elnino_sig, 'Komposit El Nino DJF 2007-2025, anomaly vs DJF 1991-2020', 'RdBu', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'Anomali curah hujan vs DJF 1991-2020 (mm/bulan)', 'both', 'rainfall_elnino_mc_recent.png'),
    (diff_elnino_anom, rain_diff_elnino_sig, 'Selisih Komposit El Nino P2 - P1, anomaly vs DJF 1991-2020', 'BrBG', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'Selisih (mm/bulan)', 'both', 'rainfall_elnino_mc_diff.png'),
]

for data, sig_mask, title, cmap, levels, ticks, cbar_label, extend, filename in plots:
    plot_and_save(
        data=data,
        sig_mask=sig_mask,
        title=title,
        cmap=cmap,
        levels=levels,
        ticks=ticks,
        cbar_label=cbar_label,
        extend=extend,
        extent=mc_extent,
        xticks=mc_xticks,
        yticks=mc_yticks,
        outfile=filename,
        figsize=(10, 6),
        ticksize=14,
        title_size=14,
        cbar_size=14,
    )


In [ ]:
export_ds = xr.Dataset(
    data_vars={
        'komposit_full': rain_clim_elnino,
        'anomali_full': rain_clim_elnino_anom,
        'pvalue_full': rain_clim_elnino_p,
        'signif_full': rain_clim_elnino_sig.astype(np.int8),
        'komposit_past': rain_past_elnino,
        'anomali_past': rain_past_elnino_anom,
        'pvalue_past': rain_past_elnino_p,
        'signif_past': rain_past_elnino_sig.astype(np.int8),
        'komposit_recent': rain_recent_elnino,
        'anomali_recent': rain_recent_elnino_anom,
        'pvalue_recent': rain_recent_elnino_p,
        'signif_recent': rain_recent_elnino_sig.astype(np.int8),
        'anomali_diff': diff_elnino_anom,
        'pvalue_diff': rain_diff_elnino_p,
        'signif_diff': rain_diff_elnino_sig.astype(np.int8),
    },
    attrs={
        'description': 'Komposit curah hujan El Nino DJF dan signifikansi statistik relatif terhadap climatology DJF 1991-2020',
        'base_climatology': 'DJF 1991-2020',
        'significance_test_full_p1_p2': '1-sample t-test vs base climatology',
        'significance_test_diff': 'Welch two-sample t-test for P2 - P1',
        'alpha': 0.05,
    },
)

if not INTERMEDIATE_PATH.exists():
    export_ds.to_netcdf(INTERMEDIATE_PATH)
    print('Saved NetCDF:', INTERMEDIATE_PATH)
else:
    print('Intermediate NetCDF already exists:', INTERMEDIATE_PATH)
